In [1]:
%load_ext autoreload
%autoreload 2
import os
if not hasattr(__builtins__, '_cwd_set'):
    os.chdir('..')
    __builtins__._cwd_set = True

In [3]:
input_path = "data/law.txt"
output_path = "data/law.txt"

with open(input_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(output_path, "w", encoding="utf-8") as f:
    for line in lines:
        if line.strip() and not line.startswith("LBK nr 282 af 17/03/2025"):
            f.write(line)

In [5]:
input_path = "data/guidelines.txt"
output_path = "data/guidelines.txt"

with open(input_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(output_path, "w", encoding="utf-8") as f:
    for line in lines:
        if line.strip() and not line.startswith("https://pro.karnovgroup.dk") and not line.startswith("12/3/25, 2:20 PM"):
            f.write(line)

In [8]:
# --- Improved: Transform law.txt to nested JSON with kapitel level ---

import re
import json

input_path = "data/law.txt"
output_path = "datajson/law.json"

with open(input_path, "r", encoding="utf-8") as f:
    lines = [line.rstrip() for line in f if line.strip()]

law_json = {}
current_kapitel = None
current_paragraph = None
current_stk = None

# Robust regexes: allow leading spaces, optional period
kapitel_re = re.compile(r"^\s*Kapitel\s+(\d+)")
paragraph_re = re.compile(r"^\s*§\s*(\d+)\.")
stk_re = re.compile(r"^\s*Stk\.\s*(\d+)\.")
numbered_item_re = re.compile(r"^\s*(\d+)\)")

for line in lines:
    sline = line.strip()
    kap_match = kapitel_re.match(sline)
    para_match = paragraph_re.match(sline)
    stk_match = stk_re.match(sline)
    numitem_match = numbered_item_re.match(sline)

    if kap_match:
        current_kapitel = f"kapitel{kap_match.group(1)}"
        law_json[current_kapitel] = {}
        current_paragraph = None
        current_stk = None
        continue

    if current_kapitel is None:
        continue  # skip lines before first Kapitel

    if para_match:
        para_num = f"§{para_match.group(1)}"
        law_json[current_kapitel][para_num] = {"title": "", "content": [], "stk": {}}
        current_paragraph = para_num
        current_stk = None
        rest = sline[para_match.end():].strip()
        if rest:
            law_json[current_kapitel][para_num]["title"] = rest
        continue

    if stk_match and current_paragraph:
        stk_num = f"Stk{stk_match.group(1)}"
        para_stk = law_json[current_kapitel][current_paragraph]["stk"]
        para_stk[stk_num] = {"text": sline, "items": []}
        current_stk = stk_num
        continue

    if numitem_match and current_paragraph:
        if current_stk:
            stk_dict = law_json[current_kapitel][current_paragraph]["stk"][current_stk]
            stk_dict["items"].append(sline)
        else:
            para_dict = law_json[current_kapitel][current_paragraph]
            if "items" not in para_dict:
                para_dict["items"] = []
            para_dict["items"].append(sline)
        continue

    if current_paragraph:
        if current_stk:
            stk_dict = law_json[current_kapitel][current_paragraph]["stk"][current_stk]
            if stk_dict["text"] == sline:
                continue
            stk_dict["text"] += " " + sline
        else:
            law_json[current_kapitel][current_paragraph]["content"].append(sline)

# Clean up: remove empty 'stk' and 'items'
for kap in law_json.values():
    for para in kap.values():
        if "stk" in para and not para["stk"]:
            del para["stk"]
        if "items" in para and not para["items"]:
            del para["items"]
        # Clean up stk items
        if "stk" in para:
            for stk in list(para["stk"].values()):
                if "items" in stk and not stk["items"]:
                    del stk["items"]

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(law_json, f, ensure_ascii=False, indent=2)